In [1]:
import os
import sys
import shutil

# ────────────────────────────────────────────────────────────────
# Clean previous working copy
# ────────────────────────────────────────────────────────────────
for path in [
    "/kaggle/working/src",
    "/kaggle/working/tft_experiment_temp",
]:
    if os.path.exists(path):
        shutil.rmtree(path)

# Remove any leftover CSVs
for f in os.listdir("/kaggle/working"):
    if f.endswith(".csv"):
        os.remove(os.path.join("/kaggle/working", f))

# ────────────────────────────────────────────────────────────────
# Copy latest project files from Kaggle Dataset
# ────────────────────────────────────────────────────────────────
print("Copying latest files from Kaggle dataset...")
!cp -r /kaggle/input/datasets/athbendre/basefiles/* /kaggle/working/

# Fix accidental nested src/src
if os.path.exists("/kaggle/working/src/src"):
    !cp -r /kaggle/working/src/src/* /kaggle/working/src/
    !rm -rf /kaggle/working/src/src

# ────────────────────────────────────────────────────────────────
# Add project to Python path
# ────────────────────────────────────────────────────────────────
if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

# Reload modules if already imported
import importlib
import src.feature_engineer
importlib.reload(src.feature_engineer)

# ────────────────────────────────────────────────────────────────
# Generate fresh synthetic dataset
# ────────────────────────────────────────────────────────────────
print("\nGenerating synthetic data...")
!python /kaggle/working/multihead_generate.py

# ────────────────────────────────────────────────────────────────
# Move generated CSVs
# ────────────────────────────────────────────────────────────────
DATA_DIR = "/kaggle/working/tft_experiment_temp/data"
os.makedirs(DATA_DIR, exist_ok=True)

!mv /kaggle/working/*.csv {DATA_DIR}/

print("\n✅ SUCCESS!")
print(f"Project files refreshed.")
print(f"Fresh synthetic data generated.")
print(f"Data location: {DATA_DIR}")

Copying latest files from Kaggle dataset...

Generating synthetic data...
Generated 5,970,360 transactions for 3250 users.
  Salaried     : 1200
  Self-Employed: 450
  Business     : 1600
Monthly summaries: 120,250 rows → /kaggle/working/monthly_summaries.csv

✅ SUCCESS!
Project files refreshed.
Fresh synthetic data generated.
Data location: /kaggle/working/tft_experiment_temp/data


# 🏦 Cashflow TFT Training Pipeline


**What this notebook does:**
1. Mounts Google Drive for persistent storage
2. Installs dependencies and clones your repo
4. Builds the PyTorch Forecasting `TimeSeriesDataSet` from your feature store
5. Trains a TFT model with static covariates (city, age, employment)
6. Evaluates with walk-forward CV — MAPE, RMSE, CI coverage
8. Saves the trained model to Google Drive

---


## Cell 1 — Mount Google Drive
Run this first every session. All models, data, and logs persist here.

In [2]:
import os

# ── Persistent directory layout on Kaggle Working Directory ─────────────
BASE_DIR     = '/kaggle/working/tft_experiment_temp'
MODEL_DIR    = f'{BASE_DIR}/models'
DATA_DIR     = f'{BASE_DIR}/data'
LOG_DIR      = f'{BASE_DIR}/logs'
COHORT_DIR   = f'{BASE_DIR}/cohort_priors'
ARTIFACT_DIR = f'{BASE_DIR}/artifacts'

for d in [MODEL_DIR, DATA_DIR, LOG_DIR, COHORT_DIR, ARTIFACT_DIR]:
    os.makedirs(d, exist_ok=True)

print('✅ Kaggle storage directories ready')
print(f'   Base   : {BASE_DIR}')
print(f'   Models : {MODEL_DIR}')
print(f'   Data   : {DATA_DIR}')


✅ Kaggle storage directories ready
   Base   : /kaggle/working/tft_experiment_temp
   Models : /kaggle/working/tft_experiment_temp/models
   Data   : /kaggle/working/tft_experiment_temp/data


## Cell 2 — Install Dependencies
Runs every session (~4 mins). Output suppressed — check the ✅ at the end.

In [3]:
%%capture install_output

# Core ML
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install pytorch-forecasting pytorch-lightning mlflow


# Data + utils
!pip install pandas numpy scikit-learn xgboost shap optuna
!pip install prophet  # kept as cold-start fallback

print('✅ All dependencies installed')

In [4]:
# Verify GPU is available
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Performance settings for Colab T4
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\n→ Training on: {DEVICE}')

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM            : 15.6 GB

→ Training on: cuda


## Cell 3 — Clone Your Repo
Pulls latest code from GitHub every session.

In [3]:
import subprocess, sys, os

GITHUB_REPO   = 'https://github.com/swaraj2442/z-business.git'  # ← change this to your repo
GITHUB_BRANCH = 'staging/cashflow'  # ← change this to your specific branch
REPO_DIR      = '/content/cashflow_pipeline'

if 'google.colab' in sys.modules:
    if os.path.exists(REPO_DIR):
        print(f'Repo exists. Pulling latest from {GITHUB_BRANCH}...')
        subprocess.run(['git', '-C', REPO_DIR, 'fetch'], capture_output=True)
        subprocess.run(['git', '-C', REPO_DIR, 'checkout', GITHUB_BRANCH], capture_output=True)
        result = subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', GITHUB_BRANCH], capture_output=True, text=True)
        print(f'Pulled latest: {result.stdout.strip()}')
    else:
        print(f'Cloning {GITHUB_BRANCH} branch from {GITHUB_REPO}...')
        result = subprocess.run(['git', 'clone', '-b', GITHUB_BRANCH, GITHUB_REPO, REPO_DIR], capture_output=True, text=True)
        print(f'Cloned: {result.stdout.strip() or result.stderr.strip()}')

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    print(f'✅ Repo ready at {REPO_DIR}')
else:
    if os.path.exists('../src'):
        sys.path.insert(0, os.path.abspath('..'))
    elif os.path.exists('./src'):
        sys.path.insert(0, os.path.abspath('.'))
    print('✅ Running locally. Using local files instead of cloning GitHub.')


Cloning staging/cashflow branch from https://github.com/swaraj2442/z-business.git...
Cloned: Cloning into '/content/cashflow_pipeline'...
fatal: could not read Username for 'https://github.com': No such device or address
✅ Repo ready at /content/cashflow_pipeline


In [5]:
# ════════════════════════════════════════════════════════
#  CELL 5 — TRAINING CONFIG (Optuna Trial 1 Best Hyperparameters)
# ════════════════════════════════════════════════════════

CFG = {
    # Data
    'transactions_csv' : '/kaggle/working/tft_experiment_temp/data/transactions_large.csv',
    'min_history_months': 6,

    # TFT architecture — Optuna Trial 1 Best Params
    'max_encoder_length'    : 12,     # how many past months TFT looks at
    'max_prediction_length' : 6,      # forecast horizon during training
    'hidden_size'           : 70,     # Optuna: Bumped from 32 → 70
    'hidden_continuous_size': 22,     # Optuna: Bumped from 16 → 22
    'attention_head_size'   : 1,      # Optuna: 1 head prevents overfitting on monthly data
    'dropout'               : 0.28,   # Optuna: Stronger regularization for larger capacity

    # Training — Optuna Trial 1 Best Params
    'batch_size'        : 32,
    'max_epochs'        : 50,
    'learning_rate'     : 0.0019, # Optuna: ~1.9e-3 (stable convergence)
    'gradient_clip_val' : 0.175,  # Optuna: ~0.175

    # Residual XGBoost
    'xgb_max_depth'           : 3,
    'xgb_n_estimators'        : 100,
    'min_months_for_residual' : 6,

    # MLflow
    'experiment_name': 'cashflow_tft_v3',
}

print('✅ Config set (Using Optuna Trial 1 Best Hyperparameters)')
print(f'   Encoder length    : {CFG["max_encoder_length"]} months')
print(f'   Prediction length : {CFG["max_prediction_length"]} months')
print(f'   Hidden size       : {CFG["hidden_size"]}')
print(f'   Continuous size   : {CFG["hidden_continuous_size"]}')
print(f'   Attention heads   : {CFG["attention_head_size"]}')
print(f'   Learning rate     : {CFG["learning_rate"]}')


✅ Config set (Using Optuna Trial 1 Best Hyperparameters)
   Encoder length    : 12 months
   Prediction length : 6 months
   Hidden size       : 70
   Continuous size   : 22
   Attention heads   : 1
   Learning rate     : 0.0019


In [6]:
import mlflow


# Set token for auth (avoids interactive prompt)

# Verify connection
MLFLOW_TRACKING_URI = 'sqlite:///mlflow.db'
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(CFG['experiment_name'])

print(f'   Tracking URI : {MLFLOW_TRACKING_URI}')
print(f'   Experiment   : {CFG["experiment_name"]}')


2026/07/26 03:34:02 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/26 03:34:02 INFO mlflow.store.db.utils: Updating database tables
2026/07/26 03:34:04 INFO mlflow.tracking.fluent: Experiment with name 'cashflow_tft_v2' does not exist. Creating a new experiment.


   Tracking URI : sqlite:///mlflow.db
   Experiment   : cashflow_tft_v2


## Cell 6 — Load Data + Feature Engineering
Runs your existing pipeline modules to build the monthly feature store.

In [8]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

from src.ingestor import load_transactions
from src.transaction_categorizer import categorize_transactions
from src.feature_engineer import build_monthly_features
from tqdm.auto import tqdm

# ── Load raw transactions ─────────────────────────────────────────
raw_df = load_transactions(CFG['transactions_csv'])
print(f'Raw transactions : {len(raw_df):,} rows')
print(f'Entities         : {raw_df["entity_id"].nunique()}')
print(f'Date range       : {raw_df["date"].min().date()} → {raw_df["date"].max().date()}')

# ── Load user profiles ────────────────────────────────────────────
profile_path = str(Path(CFG['transactions_csv']).parent / 'user_profiles.csv')
if os.path.exists(profile_path):
    profiles_df = pd.read_csv(profile_path)
    profile_map = profiles_df.set_index('entity_id')['employment_type'].to_dict()
    print(f'Profiles loaded  : {len(profiles_df):,} users')
else:
    profiles_df = None
    profile_map = {}
    print("⚠️ WARNING: user_profiles.csv not found! Features will be degraded.")

# ── Categorize ────────────────────────────────────────────────────
cat_df = categorize_transactions(raw_df)
print(f'\nCategory distribution:')
print(cat_df['category'].value_counts().head(10).to_string())

# ── Build feature store ───────────────────────────────────────────
all_features = []
skipped      = []

print("\nGrouping massive dataset (this takes a few seconds)...")
grouped_df = cat_df.groupby('entity_id')

print("Building features per user...")
for entity_id, entity_df in tqdm(grouped_df, total=len(grouped_df)):
    try:
        emp_type = profile_map.get(entity_id, 'Self-Employed')
        etype    = 'general' if emp_type == 'Salaried' else 'msme'
        feats    = build_monthly_features(
            entity_df,
            entity_id   = entity_id,
            entity_type = etype,
            profile_df  = profiles_df,
        )
        if len(feats) >= CFG['min_history_months']:
            all_features.append(feats)
        else:
            skipped.append((entity_id, len(feats), 'insufficient history'))
    except Exception as e:
        skipped.append((entity_id, 0, str(e)))

feature_store = pd.concat(all_features, ignore_index=True)
feature_store = feature_store.sort_values(['entity_id', 'period']).reset_index(drop=True)

feature_store['time_idx'] = (
    feature_store.groupby('entity_id')['period']
    .transform(lambda s: (s.dt.year - s.min().year) * 12 + (s.dt.month - s.min().month))
    .astype(int)
)

# ── Merge monthly summaries (min_balance + fixed_obligations) ─────
monthly_path = str(Path(CFG['transactions_csv']).parent / 'monthly_summaries.csv')
if os.path.exists(monthly_path):
    monthly_df = pd.read_csv(monthly_path)
    monthly_df = monthly_df[['entity_id', 'period', 'min_balance', 'fixed_obligations']]
    monthly_df['period'] = pd.to_datetime(monthly_df['period'])
    if 'fixed_obligations' in feature_store.columns:
        feature_store = feature_store.drop(columns=['fixed_obligations'])
    feature_store = feature_store.merge(monthly_df, on=['entity_id', 'period'], how='left')
    feature_store['min_balance']       = feature_store['min_balance'].fillna(0)
    feature_store['fixed_obligations'] = feature_store['fixed_obligations'].fillna(0)
    print(f'   Monthly summaries: ✅ merged (min_balance, fixed_obligations)')
else:
    feature_store['min_balance']       = 0.0
    feature_store['fixed_obligations'] = 0.0
    print('   ⚠️ monthly_summaries.csv not found — min_balance and fixed_obligations set to 0')

# ── Generate historical lag & rolling features for min_balance & fixed_obligations ──
print("   Engineering temporal lags & rolling stats for min_balance and fixed_obligations...")
g = feature_store.groupby('entity_id')

for col in ['min_balance', 'fixed_obligations']:
    for lag in [1, 2, 3, 6, 12]:
        feature_store[f'{col}_lag{lag}'] = g[col].shift(lag).fillna(0.0)
    for window in [3, 6, 12]:
        feature_store[f'{col}_roll{window}'] = (
            g[col]
            .shift(1)
            .rolling(window, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
            .fillna(0.0)
        )
    feature_store[f'{col}_std12'] = (
        g[col]
        .shift(1)
        .rolling(12, min_periods=1)
        .std()
        .reset_index(level=0, drop=True)
        .fillna(0.0)
    )

print(f'\n✅ Feature store built')
print(f'   Entities in store : {feature_store["entity_id"].nunique()}')
print(f'   Total rows        : {len(feature_store):,}')
print(f'   Columns           : {len(feature_store.columns)}')

if 'employment_type' in feature_store.columns:
    print("   Profile merge   : ✅ SUCCESS (employment_type found)")
else:
    print("   Profile merge   : ❌ ERROR (employment_type missing)")

salaried_count = feature_store[feature_store['entity_type'] == 'general']['entity_id'].nunique()
msme_count     = feature_store[feature_store['entity_type'] == 'msme']['entity_id'].nunique()
print(f'   Salaried users  : {salaried_count:,}')
print(f'   MSME users      : {msme_count:,}')


Raw transactions : 5,970,360 rows
Entities         : 3250
Date range       : 2021-06-30 → 2024-06-30
Profiles loaded  : 3,250 users

Category distribution:
category
utility         1723233
other           1207532
vendor          1199480
food             527673
transport        527019
subscription     526823
emi              117000
sales             73800
salary            43200
gst               24600

Grouping massive dataset (this takes a few seconds)...
Building features per user...


  0%|          | 0/3250 [00:00<?, ?it/s]

   Monthly summaries: ✅ merged (min_balance, fixed_obligations)
   Engineering temporal lags & rolling stats for min_balance and fixed_obligations...

✅ Feature store built
   Entities in store : 3250
   Total rows        : 119,563
   Columns           : 123
   Profile merge   : ✅ SUCCESS (employment_type found)
   Salaried users  : 1,200
   MSME users      : 2,050


In [ ]:
import os

# 1. Make sure the folder exists so it doesn't throw an error
os.makedirs('/kaggle/working/tft_experiment_temp/data', exist_ok=True)

# 2. Add the actual file name (.csv) at the end of the path!
csv_path = '/kaggle/working/tft_experiment_temp/data/feature_store.csv'

# 3. Save it
feature_store.to_csv(csv_path, index=False)
print(f'\n📂 Saved CSV for analysis to: {csv_path}')


## Cell 7 — Add Static + Known Future Features
Enriches the feature store with user profile data and calendar signals that TFT uses as covariates.

In [9]:
import os
import pandas as pd
import numpy as np

# ── Known future features (calendar signals TFT can see into the future) ──────
FESTIVAL_MONTHS = {10, 11}
GST_FILING_MONTHS = {1, 4, 7, 10}
ADVANCE_TAX_MONTHS = {3, 6, 9, 12}

feature_store['is_festival_month']    = feature_store['month'].isin(FESTIVAL_MONTHS).astype(int)
feature_store['is_gst_filing_month']  = feature_store['month'].isin(GST_FILING_MONTHS).astype(int)
feature_store['is_advance_tax_month'] = feature_store['month'].isin(ADVANCE_TAX_MONTHS).astype(int)
feature_store['quarter']              = ((feature_store['month'] - 1) // 3 + 1).astype(str)

# ── Bulletproof Profile Merge ────────────────────────────────────────────────
profile_path = '/kaggle/working/tft_experiment_temp/data/user_profiles.csv'
if os.path.exists(profile_path):
    profiles = pd.read_csv(profile_path)

    # GUARANTEE NO DOUBLE MERGE: If columns already exist, drop them first
    cols_to_drop = [c for c in profiles.columns if c != 'entity_id' and c in feature_store.columns]
    if cols_to_drop:
        feature_store = feature_store.drop(columns=cols_to_drop)

    feature_store = feature_store.merge(profiles, on='entity_id', how='left')
    print(f'✅ User profiles merged safely: {len(profiles)} profiles')
else:
    print('⚠️ WARNING: /content/data/user_profiles.csv NOT FOUND!')

# Fix MLflow illegal characters (+)
if 'age_band' in feature_store.columns:
    feature_store['age_band'] = feature_store['age_band'].astype(str).str.replace('+', '_plus')

# Ensure categorical columns are strings
for col in ['city_tier', 'age_band', 'employment_type', 'quarter', 'entity_id']:
    if col in feature_store.columns:
        feature_store[col] = feature_store[col].astype(str)

# Fill any remaining NaNs in numeric columns (Including all new lags/rolling stats)
NUMERIC_COLS = [
    'total_inflow', 'total_outflow', 'net_cashflow', 'min_balance', 'fixed_obligations',
    'emi_to_inflow_ratio', 'fixed_obligation_ratio', 'savings_rate',
    'discretionary_spend_ratio', 'upi_to_inflow_ratio',
    'net_cashflow_lag1', 'net_cashflow_lag2', 'net_cashflow_lag3',
    'net_cashflow_roll3', 'net_cashflow_roll6',
    'inflow_mom_change', 'upi_spend_mom', 'fixed_obligation_mom',
    'cost_of_living_index', 'household_size',
    'min_balance_lag1', 'min_balance_lag2', 'min_balance_lag3', 'min_balance_lag6', 'min_balance_lag12',
    'min_balance_roll3', 'min_balance_roll6', 'min_balance_roll12', 'min_balance_std12',
    'fixed_obligations_lag1', 'fixed_obligations_lag2', 'fixed_obligations_lag3', 'fixed_obligations_lag6', 'fixed_obligations_lag12',
    'fixed_obligations_roll3', 'fixed_obligations_roll6', 'fixed_obligations_roll12', 'fixed_obligations_std12',
]
for col in NUMERIC_COLS:
    if col in feature_store.columns:
        feature_store[col] = feature_store[col].replace([np.inf, -np.inf], 0).fillna(0).astype(float)

# Re-split
max_time_idx = feature_store.groupby('entity_id')['time_idx'].transform('max')
val_cutoff   = max_time_idx - CFG['max_prediction_length']
train_df = feature_store[feature_store['time_idx'] <= val_cutoff].copy()
val_df   = feature_store.copy()

print(f'\n✅ Feature enrichment and split complete')
print(f'   Train shape: {train_df.shape}')
print(f'   Val shape  : {val_df.shape}')


✅ User profiles merged safely: 3250 profiles

✅ Feature enrichment and split complete
   Train shape: (100063, 123)
   Val shape  : (119563, 123)


In [10]:
import numpy as np

# 1. Guarantee no infinities or NaNs exist in the parent dataframe
feature_store = feature_store.replace([np.inf, -np.inf], 0).fillna(0)

# 2. Recreate train_df and val_df from the CLEANED parent dataframe
max_time_idx = feature_store.groupby('entity_id')['time_idx'].transform('max')
val_cutoff   = max_time_idx - CFG['max_prediction_length']

train_df = feature_store[feature_store['time_idx'] <= val_cutoff].copy()
val_df   = feature_store.copy()

print(f"✅ Cleaned datasets recreated! Train: {len(train_df)}, Val: {len(val_df)}")


✅ Cleaned datasets recreated! Train: 100063, Val: 119563


In [11]:
import numpy as np

# Fix infinities
feature_store = feature_store.replace([np.inf, -np.inf], 0).fillna(0)

# Fix MLflow illegal characters (+)
feature_store['age_band'] = feature_store['age_band'].str.replace('+', '_plus')

# Re-split
max_time_idx = feature_store.groupby('entity_id')['time_idx'].transform('max')
val_cutoff   = max_time_idx - CFG['max_prediction_length']
train_df = feature_store[feature_store['time_idx'] <= val_cutoff].copy()
val_df   = feature_store.copy()

cols = [
    "min_balance",
    "min_balance_lag1",
    "min_balance_lag2",
    "min_balance_roll3",
    "fixed_obligations",
    "fixed_obligations_lag1",
]

print(feature_store[cols].head(15))


    min_balance  min_balance_lag1  min_balance_lag2  min_balance_roll3  \
0     201470.65              0.00              0.00           0.000000   
1     201470.65         201470.65              0.00      201470.650000   
2     223671.96         201470.65         201470.65      201470.650000   
3     237112.52         223671.96         201470.65      208871.086667   
4     266076.87         237112.52         223671.96      220751.710000   
5     290076.41         266076.87         237112.52      242287.116667   
6     323358.92         290076.41         266076.87      264421.933333   
7     333699.92         323358.92         290076.41      293170.733333   
8     325237.19         333699.92         323358.92      315711.750000   
9     325237.19         325237.19         333699.92      327432.010000   
10    325327.33         325237.19         325237.19      328058.100000   
11    346030.98         325327.33         325237.19      325267.236667   
12    353962.67         346030.98     

In [ ]:
cols = [
    "min_balance",
    "min_balance_lag1",
    "min_balance_lag2",
    "min_balance_roll3",
    "fixed_obligations",
    "fixed_obligations_lag1",
]

print(feature_store[cols].head(15))

## Cell 8 — Train / Val Split + PyTorch Forecasting Dataset
Builds the `TimeSeriesDataSet` that TFT expects. Chronological split — no leakage.

In [14]:
!pip install "optuna-integration[pytorch_lightning]"


In [17]:
# CELL 8 — True Multi-Output TimeSeriesDataSet
# ══════════════════════════════════════════════════════════════════

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer, MultiNormalizer
from pytorch_forecasting.metrics import QuantileLoss, MultiLoss

# ── Chronological split ──────────────────────────────
max_time_idx = feature_store.groupby('entity_id')['time_idx'].transform('max')
val_cutoff   = max_time_idx - CFG['max_prediction_length']
train_df     = feature_store[feature_store['time_idx'] <= val_cutoff].copy()
val_df       = feature_store.copy()

print(f'Train rows : {len(train_df):,}')
print(f'Val rows   : {len(val_df):,}')

# ── Feature groups ───────────────────────────────────
TIME_VARYING_KNOWN_REALS = [
    'month', 'is_festival_month', 'is_gst_filing_month', 'is_advance_tax_month',
]
TIME_VARYING_KNOWN_CATEGORICALS = ['quarter']

STATIC_CATEGORICALS = ['employment_type', 'city_tier', 'age_band']
STATIC_REALS        = ['household_size', 'cost_of_living_index']

# ── 4 Targets for True Multi-Output ────────────────────────────────
TARGET_COLS = [
    'total_inflow',
    'total_outflow',
    'min_balance',
    'fixed_obligations'
]

# We must remove targets from the TVU reals list
BASE_TVU_REALS = [
    'emi_to_inflow_ratio',
    'fixed_obligation_ratio',
    'discretionary_spend_ratio',
    'net_cashflow_lag1',
    'net_cashflow_lag2',
    'net_cashflow_lag3',
    'net_cashflow_roll3',
    'net_cashflow_roll6',
    'inflow_mom_change',
    'upi_spend_mom',
    'fixed_obligation_mom',
    'upi_to_inflow_ratio',
    'net_cashflow',
    'savings_rate',
    'total_inflow',
    'total_outflow',
    'net_cashflow_lag6',
    'net_cashflow_lag12',
    'net_cashflow_roll12',
    'total_inflow_lag1',
    'total_inflow_lag2',
    'total_inflow_lag3',
    'total_inflow_lag6',
    'total_inflow_lag12',
    'total_outflow_lag1',
    'total_outflow_lag2',
    'total_outflow_lag3',
    'total_outflow_lag6',
    'total_outflow_lag12',
    'total_inflow_roll3',
    'total_inflow_roll6',
    'total_inflow_roll12',
    'total_outflow_roll3',
    'total_outflow_roll6',
    'total_outflow_roll12',
    'inflow_volatility6',
    'total_inflow_std12',
    'total_outflow_std12',
    # ── NEW: Historical Lags & Rolling Stats for min_balance & fixed_obligations ──
    'min_balance_lag1',
    'min_balance_lag2',
    'min_balance_lag3',
    'min_balance_lag6',
    'min_balance_lag12',
    'min_balance_roll3',
    'min_balance_roll6',
    'min_balance_roll12',
    'min_balance_std12',
    'fixed_obligations_lag1',
    'fixed_obligations_lag2',
    'fixed_obligations_lag3',
    'fixed_obligations_lag6',
    'fixed_obligations_lag12',
    'fixed_obligations_roll3',
    'fixed_obligations_roll6',
    'fixed_obligations_roll12',
    'fixed_obligations_std12',
]
# Remove any target columns that might accidentally be in TVU reals
TVU_REALS = [c for c in BASE_TVU_REALS if c not in TARGET_COLS]

def _filter_existing(cols):
    return [c for c in cols if c in feature_store.columns]

tvu = _filter_existing(TVU_REALS)
tvk = _filter_existing(TIME_VARYING_KNOWN_REALS)
sc  = _filter_existing(STATIC_CATEGORICALS)
sr  = _filter_existing(STATIC_REALS)
tkc = _filter_existing(TIME_VARYING_KNOWN_CATEGORICALS)

# Create MultiNormalizer with a GroupNormalizer for each target
target_normalizer = MultiNormalizer([
    GroupNormalizer(groups=['entity_id']) for _ in TARGET_COLS
])

train_ds = TimeSeriesDataSet(
    train_df,
    time_idx                        = 'time_idx',
    target                          = TARGET_COLS,
    group_ids                       = ['entity_id'],
    max_encoder_length              = CFG['max_encoder_length'],
    max_prediction_length           = CFG['max_prediction_length'],
    static_categoricals             = sc,
    static_reals                    = sr,
    time_varying_known_reals        = tvk,
    time_varying_known_categoricals = tkc,
    time_varying_unknown_reals      = tvu,
    target_normalizer               = target_normalizer,
    add_relative_time_idx           = True,
    add_target_scales               = True,
    add_encoder_length              = True,
    allow_missing_timesteps         = True,
)

val_ds = TimeSeriesDataSet.from_dataset(
    train_ds, val_df, predict=True, stop_randomization=True
)

train_loader = train_ds.to_dataloader(
    train=True, batch_size=CFG['batch_size'], num_workers=0
)
val_loader = val_ds.to_dataloader(
    train=False, batch_size=CFG['batch_size'], num_workers=0
)

print(f'\n{"═"*60}')
print(f'  Dataset summary (True Multi-Output: {len(TARGET_COLS)} targets)')
print(f'{"═"*60}')
print(f'  Target columns       : {TARGET_COLS}')
print(f'  Static categoricals  : {train_ds.static_categoricals}')
print(f'  Static reals         : {train_ds.static_reals}')
print(f'  Time varying known   : {train_ds.time_varying_known_reals}')
print(f'  Time varying unknown : {len(train_ds.time_varying_unknown_reals)} features')
print(f'  Train loader batches : {len(train_loader)}')
print(f'  Val loader batches   : {len(val_loader)}')


Train rows : 100,063
Val rows   : 119,563

════════════════════════════════════════════════════════════
  Dataset summary (True Multi-Output: 4 targets)
════════════════════════════════════════════════════════════
  Target columns       : ['total_inflow', 'total_outflow', 'min_balance', 'fixed_obligations']
  Static categoricals  : ['employment_type', 'city_tier', 'age_band']
  Static reals         : ['household_size', 'cost_of_living_index']
  Time varying known   : ['month', 'is_festival_month', 'is_gst_filing_month', 'is_advance_tax_month']
  Time varying unknown : 54 features
  Train loader batches : 1400
  Val loader batches   : 102


In [ ]:
print(feature_store.filter(regex="net_cashflow").columns.tolist())

In [ ]:
missing = [c for c in BASE_TVU_REALS if c not in feature_store.columns]
print("Missing:", missing)

In [ ]:
print("TVU_REALS:")
print(tvu)

## Cell 9 - Optuna

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 9 — Optuna Hyperparameter Tuning
# ══════════════════════════════════════════════════════════════════

import optuna
from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters
from pytorch_forecasting.metrics import QuantileLoss, MultiLoss

print(f'\n{"═"*60}')
print(" Running Optuna Hyperparameter Search")
print(f'{"═"*60}')

# Multi-target quantile loss
multi_loss = MultiLoss([
    QuantileLoss([0.05, 0.50, 0.95])
    for _ in TARGET_COLS
])

# Hyperparameter search
study = optimize_hyperparameters(
    train_loader,
    val_loader,
    model_path="optuna_multitarget_v6",

    n_trials=75,
    max_epochs=20,

    hidden_size_range=(16, 64),
    hidden_continuous_size_range=(8, 32),
    attention_head_size_range=(1, 4),
    learning_rate_range=(5e-4, 3e-2),
    dropout_range=(0.20, 0.50),
    gradient_clip_val_range=(0.01, 1.0),

    loss=multi_loss,

    trainer_kwargs=dict(
        accelerator="gpu",
        devices=1,
    ),
)

print(f"\n✅ Optuna Search Complete!")
print(f"Best Validation Loss: {study.best_value:,.2f}")

print("\nBest Hyperparameters:")
for k, v in study.best_trial.params.items():
    print(f"  {k}: {v}")

# Save study for future debugging
optuna.study.Study.trials_dataframe(study).to_csv(
    "optuna_trials.csv",
    index=False
)

print("\n📄 Trial history saved to: optuna_trials.csv")

[I 2026-07-26 05:58:04,242] A new study created in memory with name: no-name-d6545b16-9ecc-4f06-921f-16666156f5a8
/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/models/temporal_fusion_transformer/tuning.py:176: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  gradient_clip_val = trial.suggest_loguniform(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.



════════════════════════════════════════════════════════════
 Running Optuna Hyperparameter Search
════════════════════════════════════════════════════════════


/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/models/temporal_fusion_transformer/tuning.py:202: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  dropout=trial.suggest_uniform("dropout", *dropout_range),
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.
Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. Y

Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=100` reached.
Restoring states from the checkpoint path at /kaggle/working/.lr_find_bca2ca84-fded-4c2d-9bb8-56ef318c955e.ckpt
Restored all states from the checkpoint at /kaggle/working/.lr_find_bca2ca84-fded-4c2d-9bb8-56ef318c955e.ckpt
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
Learning rate set to 0.000817237251231966
[I 2026-07-26 05:58:28,263] Using learning rate of 0.000851
/usr/local/lib/python3.12/dist-packages/pytorch_forecasting/models/temporal_fusion_transformer/tuning.py:252: FutureWarning: su

## Cell 10 — Train
This is where the GPU earns its keep. Expected time on T4: ~3–8 mins for 50 epochs at beta scale.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 10 — Train Final TFT Model Using Optuna's Best Hyperparameters
# ══════════════════════════════════════════════════════════════════

import time
import lightning.pytorch as pl
from lightning.pytorch.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    LearningRateMonitor,
)
from lightning.pytorch.loggers import MLFlowLogger, TensorBoardLogger
from pytorch_forecasting.metrics import QuantileLoss, MultiLoss
import os
import pandas as pd

print(f'\n{"═"*60}')
print(f'  Training: True Multi-Output TFT (With Optuna Best Params)')
print(f'{"═"*60}')

# ── Auto-apply Optuna hyperparameters if study exists ──
if 'study' in locals():
    best_p = study.best_trial.params
    print("⚡ Wiring Optuna best hyperparameters into CFG:")
    CFG['learning_rate']          = best_p.get('learning_rate', CFG['learning_rate'])
    CFG['hidden_size']            = best_p.get('hidden_size', CFG['hidden_size'])
    CFG['attention_head_size']    = best_p.get('attention_head_size', CFG['attention_head_size'])
    CFG['dropout']                = best_p.get('dropout', CFG['dropout'])
    CFG['hidden_continuous_size'] = best_p.get('hidden_continuous_size', CFG['hidden_continuous_size'])
    CFG['gradient_clip_val']      = best_p.get('gradient_clip_val', CFG.get('gradient_clip_val', 0.1))
    for k in ['hidden_size', 'hidden_continuous_size', 'attention_head_size', 'learning_rate', 'dropout']:
        print(f'    • {k}: {CFG[k]}')
else:
    print("ℹ️ No Optuna study found in memory. Using fallback CFG parameters.")

# ── Loggers ───────────────────────────────────────────────────────
mlflow_logger = MLFlowLogger(
    experiment_name='cashflow_multitarget_v3',
    run_name='tft_multi_output_optuna',
    tracking_uri=MLFLOW_TRACKING_URI,
    tags={'targets': "all_4_targets"},
)

tb_logger = TensorBoardLogger(
    save_dir="lightning_logs",
    name="tft_multi_output",
)

# ── Callbacks ─────────────────────────────────────────────────────
ckpt_cb = ModelCheckpoint(
    dirpath=f'{MODEL_DIR}/tft_multi_output',
    monitor='val_loss',
    save_top_k=1,
    mode='min',
)

early_cb = EarlyStopping(
    monitor='val_loss',
    patience=5,
    mode='min'
)

lr_monitor = LearningRateMonitor(logging_interval="epoch")

# ── Loss ──────────────────────────────────────────────────────────
multi_loss = MultiLoss([
    QuantileLoss([0.05, 0.50, 0.95])
    for _ in TARGET_COLS
])

# ── Model ─────────────────────────────────────────────────────────
tft = TemporalFusionTransformer.from_dataset(
    train_ds,
    learning_rate=CFG['learning_rate'],
    hidden_size=CFG['hidden_size'],
    attention_head_size=CFG['attention_head_size'],
    dropout=CFG['dropout'],
    hidden_continuous_size=CFG['hidden_continuous_size'],
    loss=multi_loss,
    log_interval=10,
    log_val_interval=1,
    reduce_on_plateau_patience=3,
)

# ── Trainer ───────────────────────────────────────────────────────
trainer = pl.Trainer(
    min_epochs=10,
    max_epochs=CFG['max_epochs'],
    gradient_clip_val=CFG.get('gradient_clip_val', 0.1),
    accelerator='gpu' if DEVICE == 'cuda' else 'cpu',
    devices=1,
    logger=[mlflow_logger, tb_logger],
    callbacks=[
        ckpt_cb,
        early_cb,
        lr_monitor,
    ],
    log_every_n_steps=10,
    enable_progress_bar=True,
)
print("\nMODEL CONFIGURATION")
print("-"*60)
print(f"learning_rate           : {CFG['learning_rate']}")
print(f"hidden_size             : {CFG['hidden_size']}")
print(f"hidden_continuous_size  : {CFG['hidden_continuous_size']}")
print(f"attention_head_size     : {CFG['attention_head_size']}")
print(f"dropout                 : {CFG['dropout']}")
print(f"gradient_clip_val       : {CFG.get('gradient_clip_val',0.1)}")
print(f"max_epochs              : {CFG['max_epochs']}")
print("-"*60)
# ── Train ─────────────────────────────────────────────────────────
t0 = time.time()
trainer.fit(tft, train_loader, val_loader)


# Save Lightning metrics
if tb_logger.log_dir:
    metrics_path = os.path.join(tb_logger.log_dir, "metrics.csv")
    if os.path.exists(metrics_path):
        metrics = pd.read_csv(metrics_path)
        print(f"\n📊 Training history saved: {metrics_path}")

        cols = [c for c in ["epoch", "step", "train_loss_epoch", "val_loss"] if c in metrics.columns]
        if cols:
            print(metrics[cols].dropna(how="all").tail())

best = TemporalFusionTransformer.load_from_checkpoint(
    ckpt_cb.best_model_path
)
best.eval()
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Current epoch      : {trainer.current_epoch}")
print(f"Best checkpoint    : {ckpt_cb.best_model_path}")
print(f"Best val_loss      : {ckpt_cb.best_model_score:.4f}")
print(f"Stopped early      : {trainer.current_epoch < CFG['max_epochs']}")
print("="*60)

print(f'\n✅ Multi-Output Model Trained')
print(f'  val_loss: {ckpt_cb.best_model_score.item():,.1f}  |  {(time.time()-t0)/60:.1f} min')
print(f'  Checkpoint at: {ckpt_cb.best_model_path}')

# ── SANITY CHECK ──────────────────────────────────────────────────
print("\n" + "═"*60)
print("  SANITY CHECK: Quantiles & Prediction Tensor Shape")
print("═"*60)

for i, tcol in enumerate(TARGET_COLS):
    q_list = best.loss[i].quantiles if hasattr(best.loss, '__getitem__') else best.loss.quantiles
    print(f"  Target [{tcol}] configured quantiles: {q_list}")

_sample_preds = best.predict(
    val_loader,
    mode='quantiles',
    return_y=False,
)

print(f"  Prediction tensor shape (Target 0): {_sample_preds[0].shape}")
print("  Expected shape: (batch, horizon, len(quantiles))")

print("\n📈 TensorBoard logs saved to: lightning_logs/")
print("Run:")
print("tensorboard --logdir lightning_logs")

## Cell 11 — Load Best Checkpoint + Evaluate
Loads the best checkpoint (lowest val loss) and computes MAPE, RMSE, and CI coverage.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 11 — TRAIN + VALIDATION METRICS
# ══════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def evaluate_loader(model, loader, dataset_name):
    print(f'\n{"═"*70}')
    print(f'  {dataset_name.upper()} METRICS')
    print(f'{"═"*70}')

    results = model.predict(loader, mode="quantiles", return_y=True)

    preds_list = results.output
    y_true_list = results.y[0]

    metrics = {}

    for i, target_col in enumerate(TARGET_COLS):

        preds = preds_list[i].cpu().numpy()
        y_true = y_true_list[i].cpu().numpy().flatten()

        # ----- Quantile indices -----
        q_vals = (
            model.loss[i].quantiles
            if hasattr(model.loss, "__getitem__")
            else model.loss.quantiles
        )

        idx_p50 = (
            q_vals.index(0.50)
            if 0.50 in q_vals
            else (
                q_vals.index(0.5)
                if 0.5 in q_vals
                else len(q_vals) // 2
            )
        )

        idx_low = 0
        idx_high = len(q_vals) - 1

        p_low = preds[:, :, idx_low].flatten()
        p50 = preds[:, :, idx_p50].flatten()
        p_high = preds[:, :, idx_high].flatten()

        wmape = (
            np.sum(np.abs(y_true - p50))
            / (np.sum(np.abs(y_true)) + 1e-8)
            * 100
        )

        rmse = np.sqrt(mean_squared_error(y_true, p50))
        mae = mean_absolute_error(y_true, p50)
        r2 = r2_score(y_true, p50)
        coverage = np.mean(
            (y_true >= p_low) & (y_true <= p_high)
        ) * 100

        print(f"\n--> {target_col}")
        print(f"WMAPE      : {wmape:.2f}%")
        print(f"RMSE       : {rmse:,.2f}")
        print(f"MAE        : {mae:,.2f}")
        print(f"R²         : {r2:.4f}")
        print(f"CI Coverage: {coverage:.2f}%")

        metrics[target_col] = {
            "WMAPE%": wmape,
            "RMSE": rmse,
            "MAE": mae,
            "R²": r2,
            "CI Coverage%": coverage,
        }

    summary = pd.DataFrame(metrics).T

    print(f'\n{"-"*70}')
    print(f'{dataset_name.upper()} SUMMARY')
    print(f'{"-"*70}')
    print(summary.to_string(float_format=lambda x: f"{x:,.2f}"))

    return summary


# ==========================================================
# Evaluate BOTH datasets
# ==========================================================

train_summary = evaluate_loader(best, train_loader, "Training")
val_summary = evaluate_loader(best, val_loader, "Validation")

In [28]:
import os

for root, dirs, files in os.walk("lightning_logs"):
    for f in files:
        if f == "metrics.csv":
            print(os.path.join(root, f))

## Cell 12 — Attention + Variable Importance
TFT's built-in interpretability — see which features and which past months it's attending to.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ── Variable importance (which features does TFT rely on most?) ───────────────
# Get the raw numerical interpretation scores
interpretation = best.interpret_output(
    best.predict(val_loader, mode='raw', return_x=True)[0],
    reduction='sum',
)

# 1. Encoder Variables (Past Features)
enc_importance_scores = interpretation['encoder_variables'].cpu().numpy()
enc_feature_names = best.encoder_variables

enc_df = pd.DataFrame({
    'feature': enc_feature_names,
    'importance': enc_importance_scores
}).sort_values('importance', ascending=True)

# 2. Decoder Variables (Future Features)
dec_importance_scores = interpretation['decoder_variables'].cpu().numpy()
dec_feature_names = best.decoder_variables

dec_df = pd.DataFrame({
    'feature': dec_feature_names,
    'importance': dec_importance_scores
}).sort_values('importance', ascending=True)

# ── Plotting ────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Plot Encoder Importance
ax1.barh(enc_df['feature'], enc_df['importance'], color='skyblue')
ax1.set_title("Encoder (Past) Feature Importance")
ax1.set_xlabel("Importance Score")

# Plot Decoder Importance
ax2.barh(dec_df['feature'], dec_df['importance'], color='salmon')
ax2.set_title("Decoder (Future) Feature Importance")
ax2.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()

# P.S. PyTorch Forecasting auto-plotter:
# best.plot_interpretation(interpretation)


## Cell 13 — XGBoost Personal Residual Layer
Trains a per-user XGBoost on TFT's residuals. Corrects systematic errors for individual users.

In [ ]:
# (Residual XGBoost code commented out pending multi-output architectural update)
# You can implement a single multi-output XGBoost or 6 independent XGBoosts here later.
# ══════════════════════════════════════════════════════════════════
# CELL 13 — XGBoost Per-User Residual Correction Layer
# ══════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

print(f'\n{"═"*60}')
print('  CELL 13 — XGBoost Residual Correction')
print(f'{"═"*60}')

# ── Step 1: Get TFT predictions on the full TRAINING set ──────────
# We need residuals = (y_true - y_pred) on the train set
# to train the XGBoost residual correctors.

# Re-use the train loader for residual extraction
train_loader_for_residuals = train_ds.to_dataloader(
    train=False, batch_size=CFG['batch_size'] * 2,
    num_workers=2, persistent_workers=True
)

print('Extracting TFT residuals on training set...')
raw_preds_train, idx_train = best.predict(
    train_loader_for_residuals, mode='raw', return_index=True
)

# raw_preds_train is a list of tensors, one per target
# Each tensor: (N, pred_len, n_quantiles)
TARGET_COLS_PRED = ['total_inflow', 'total_outflow', 'min_balance', 'fixed_obligations']

# ── Step 2: Build residual features ───────────────────────────────
# Features available at inference time (known future + static)
RESIDUAL_FEATURES = [
    'month', 'is_festival_month', 'is_gst_filing_month', 'is_advance_tax_month',
    'household_size', 'cost_of_living_index',
]

def build_residual_df(preds_raw, idx, dataset_df, target_cols):
    """Build a flat DataFrame with (entity_id, time_idx, tft_pred, y_true) per target."""
    q_vals = best.loss[0].quantiles if hasattr(best.loss, '__getitem__') else best.loss.quantiles
    idx_p50 = q_vals.index(0.50) if 0.50 in q_vals else (q_vals.index(0.5) if 0.5 in q_vals else len(q_vals) // 2)
    records = []
    for i, (entity_id, t_start) in enumerate(zip(idx['entity_id'], idx['time_idx'])):
        for step in range(CFG['max_prediction_length']):
            t_idx = t_start + step
            row_mask = (
                (dataset_df['entity_id'] == entity_id) &
                (dataset_df['time_idx'] == t_idx)
            )
            row = dataset_df[row_mask]
            if len(row) == 0:
                continue
            rec = {'entity_id': entity_id, 'time_idx': int(t_idx)}
            for feat in RESIDUAL_FEATURES:
                if feat in row.columns:
                    rec[feat] = float(row[feat].values[0])
            for t_idx_col, tcol in enumerate(target_cols):
                # p50 prediction (median quantile, index 1 out of [p10, p50, p90])
                rec[f'tft_pred_{tcol}'] = float(preds_raw[t_idx_col][i, step, idx_p50])
                if tcol in row.columns:
                    rec[f'y_true_{tcol}'] = float(row[tcol].values[0])
                    rec[f'residual_{tcol}'] = rec[f'y_true_{tcol}'] - rec[f'tft_pred_{tcol}']
            records.append(rec)
    return pd.DataFrame(records)

print('Building residual DataFrame (this may take ~30s)...')
residual_df_train = build_residual_df(
    raw_preds_train, idx_train, train_df, TARGET_COLS_PRED
)
print(f'  Residual train rows: {len(residual_df_train):,}')

# ── Step 3: Train one XGBoost per target on the residuals ─────────
# We train on: Inflow residuals and Outflow residuals (the 2 primary targets).
# min_balance and fixed_obligations are harder to correct without extra signals,
# but we include them too for completeness.

xgb_models = {}
xgb_feature_cols = RESIDUAL_FEATURES + [f'tft_pred_{tcol}' for tcol in TARGET_COLS_PRED]
xgb_feature_cols = [c for c in xgb_feature_cols if c in residual_df_train.columns]

for tcol in TARGET_COLS_PRED:
    residual_col = f'residual_{tcol}'
    if residual_col not in residual_df_train.columns:
        print(f'  Skipping {tcol} — residual column missing.')
        continue
    sub = residual_df_train.dropna(subset=[residual_col] + xgb_feature_cols)
    X_train_xgb = sub[xgb_feature_cols].values
    y_train_xgb = sub[residual_col].values

    model = xgb.XGBRegressor(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbosity=0,
    )
    model.fit(X_train_xgb, y_train_xgb)
    xgb_models[tcol] = model
    print(f'  ✅ Trained XGBoost residual corrector for: {tcol}')

print(f'\n✅ XGBoost residual models trained for {len(xgb_models)} targets.')


## Cell 14 — Hierarchical Blending + Final Inference
Combines TFT global forecast + XGBoost residual correction using history-length-based weights.

In [ ]:
# (Multi-target hierarchical predict commented out pending multi-output architectural update)
# Inference is now much simpler: just one `best.predict(...)` call returning all 6 targets.
# ══════════════════════════════════════════════════════════════════
# CELL 14 — Hierarchical Blending + Final Inference
# ══════════════════════════════════════════════════════════════════

print(f'\n{"═"*60}')
print('  CELL 14 — Final Hierarchical Inference')
print(f'{"═"*60}')

# ── Step 1: Get TFT predictions on the validation set ─────────────
print('Getting TFT predictions on validation set...')
raw_preds_val, idx_val = best.predict(
    val_loader, mode='raw', return_index=True
)

# ── Step 2: Build val residual feature dataframe ──────────────────
print('Building validation feature DataFrame for XGBoost correction...')
residual_df_val = build_residual_df(
    raw_preds_val, idx_val, val_df, TARGET_COLS_PRED
)
print(f'  Validation rows: {len(residual_df_val):,}')

# ── Step 3: Apply XGBoost residual corrections ────────────────────
# Corrected Prediction = TFT p50 + XGBoost(residual_features)
corrected_df = residual_df_val.copy()

for tcol in TARGET_COLS_PRED:
    tft_pred_col = f'tft_pred_{tcol}'
    corr_col = f'corrected_{tcol}'
    if tcol in xgb_models and tft_pred_col in corrected_df.columns:
        X_val_xgb = corrected_df[xgb_feature_cols].fillna(0).values
        residual_correction = xgb_models[tcol].predict(X_val_xgb)
        corrected_df[corr_col] = corrected_df[tft_pred_col] + residual_correction
    else:
        corrected_df[corr_col] = corrected_df.get(tft_pred_col, 0)

# ── Step 4: Derive net_cashflow and savings_rate ───────────────────
# This is the KEY accounting consistency step!
# net_cashflow = corrected_inflow - corrected_outflow
# savings_rate = net_cashflow / corrected_inflow (clamped to [-1, 1])

corrected_df['derived_net_cashflow'] = (
    corrected_df['corrected_total_inflow'] - corrected_df['corrected_total_outflow']
)
corrected_df['derived_savings_rate'] = (
    corrected_df['derived_net_cashflow'] /
    (corrected_df['corrected_total_inflow'].abs() + 1e-8)
).clip(-1, 1)

# ── Step 5: Final evaluation — Corrected vs Ground Truth ──────────
print(f'\n{"─"*60}')
print('  FINAL METRICS — Corrected (TFT + XGBoost) vs Ground Truth')
print(f'{"─"*60}')

ALL_EVAL_COLS = {
    'total_inflow':     'corrected_total_inflow',
    'total_outflow':    'corrected_total_outflow',
    'min_balance':      'corrected_min_balance',
    'fixed_obligations':'corrected_fixed_obligations',
    'net_cashflow':     'derived_net_cashflow',
    'savings_rate':     'derived_savings_rate',
}

final_metrics = {}
for tcol, pred_col in ALL_EVAL_COLS.items():
    true_col = f'y_true_{tcol}' if tcol in TARGET_COLS_PRED else None
    if true_col is None or true_col not in corrected_df.columns:
        # For derived metrics, check val_df directly
        if tcol not in val_df.columns:
            continue

    sub = corrected_df.dropna(subset=[pred_col])
    if true_col and true_col in sub.columns:
        y_true = sub[true_col].values
    else:
        continue

    y_pred = sub[pred_col].values

    wmape = np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + 1e-8) * 100
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    mae   = mean_absolute_error(y_true, y_pred)
    r2    = r2_score(y_true, y_pred)

    print(f'\n  [{tcol}]')
    print(f'    WMAPE : {wmape:.1f}%')
    print(f'    RMSE  : {rmse:,.0f}')
    print(f'    MAE   : {mae:,.0f}')
    print(f'    R²    : {r2:+.3f}')

    final_metrics[tcol] = {'wmape': wmape, 'rmse': rmse, 'mae': mae, 'r2': r2}

print(f'\n{"═"*60}')
print('  SUMMARY TABLE')
print(f'{"═"*60}')
final_df = pd.DataFrame(final_metrics).T[['wmape', 'rmse', 'mae', 'r2']]
final_df.columns = ['WMAPE%', 'RMSE', 'MAE', 'R²']
print(final_df.to_string(float_format=lambda x: f'{x:,.2f}'))

# ── Step 6: Save corrected predictions to Google Drive ────────────
out_path = f'{MODEL_DIR}/final_corrected_predictions.csv'
corrected_df.to_csv(out_path, index=False)
print(f'\n✅ Final corrected predictions saved to: {out_path}')


## Cell 15 — Save Everything to Drive + Log Final Artifacts

In [ ]:
import shutil
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M')

# ── Save TFT model ────────────────────────────────────────────────────────────
tft_save_path = f'{MODEL_DIR}/tft_model_{timestamp}.ckpt'
shutil.copy(trainer.checkpoint_callback.best_model_path, tft_save_path)
print(f'✅ TFT model saved: {tft_save_path}')

# ── Save training dataset params (needed to reconstruct dataset for inference)
import pickle
dataset_params_path = f'{MODEL_DIR}/training_dataset_params_{timestamp}.pkl'
with open(dataset_params_path, 'wb') as f:
    pickle.dump(training_dataset.get_parameters(), f)
print(f'✅ Dataset params saved: {dataset_params_path}')

# ── Save feature store ────────────────────────────────────────────────────────
fs_save_path = f'{DATA_DIR}/feature_store_{timestamp}.parquet'
feature_store.to_parquet(fs_save_path, index=False)
print(f'✅ Feature store saved: {fs_save_path}')

# ── Log to MLflow ─────────────────────────────────────────────────────────────
with mlflow.start_run(run_id=mlflow_logger.run_id):
    mlflow.log_artifact(tft_save_path, artifact_path='model')
    mlflow.log_artifact(dataset_params_path, artifact_path='model')
    mlflow.log_param('model_timestamp', timestamp)
    mlflow.log_param('n_entities_trained', feature_store['entity_id'].nunique())
    mlflow.log_param('n_residual_models', len(residual_models))

print()
print('── Summary ─────────────────────────────────────────────────')
print(f'  TFT val MAPE     : {metrics["val_mape"]:.1f}%')
print(f'  TFT val RMSE     : {metrics["val_rmse"]:,.0f}')
print(f'  CI Coverage      : {metrics["val_ci_coverage"]:.1f}%')
print(f'  Residual models  : {len(residual_models)} users')
print(f'  MLflow run ID    : {mlflow_logger.run_id}')
print('────────────────────────────────────────────────────────────')

## Cell 16 — Reload Model (Next Session)
Run this instead of training to load a previously saved model from Drive.

In [ ]:
# ── Uncomment and run this to load a saved model without retraining ────────────

# import pickle, joblib, glob
#
# # Find latest model
# model_files = sorted(glob.glob(f'{MODEL_DIR}/tft_model_*.ckpt'))
# latest_model_path = model_files[-1]
#
# # Find matching dataset params
# ts = latest_model_path.split('tft_model_')[1].replace('.ckpt', '')
# dataset_params_path = f'{MODEL_DIR}/training_dataset_params_{ts}.pkl'
#
# # Rebuild dataset from saved params (needed for inference)
# with open(dataset_params_path, 'rb') as f:
#     dataset_params = pickle.load(f)
#
# # Load TFT
# best_model = TemporalFusionTransformer.load_from_checkpoint(latest_model_path)
# best_model.eval()
#
# # Load residual models
# residual_models = {}
# for path in glob.glob(f'{RESIDUAL_MODEL_DIR}/residual_*.joblib'):
#     entity_id = path.split('residual_')[1].replace('.joblib', '')
#     residual_models[entity_id] = joblib.load(path)
#
# print(f'✅ Loaded TFT from: {latest_model_path}')
# print(f'   Residual models : {len(residual_models)}')

print('Uncomment the block above to load a saved model from Drive.')

In [ ]:
import os
import warnings
import kagglehub
import pandas as pd
import numpy as np
import lightning.pytorch as pl
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer, QuantileLoss

warnings.filterwarnings('ignore')

print("1. Downloading Kaggle Dataset...")
path = kagglehub.dataset_download("khushikyad001/personal-finance-tracker-dataset")

# Find the exact CSV file in the downloaded folder
csv_file = [os.path.join(r, f) for r, d, files in os.walk(path) for f in files if f.endswith('.csv')][0]
df = pd.read_csv(csv_file)
print(f"Dataset loaded with {len(df)} rows.")

print("\n2. Feature Engineering for TFT...")
# Convert date and sort chronologically per user
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['user_id', 'date']).reset_index(drop=True)

# TFT needs a continuous integer time index (0, 1, 2, 3...) per user
df['time_idx'] = df.groupby('user_id').cumcount()

# Ensure categoricals are properly typed as strings
categorical_cols = ['financial_scenario', 'income_type', 'category', 'cash_flow_status', 'financial_stress_level']
for col in categorical_cols:
    df[col] = df[col].astype(str)
df['user_id'] = df['user_id'].astype(str)

# TFT requires entities to have enough history to look back.
# We'll filter for users with at least 4 historical records.
counts = df['user_id'].value_counts()
valid_users = counts[counts >= 4].index
df = df[df['user_id'].isin(valid_users)]
print(f"Filtered to {len(df)} rows ({len(valid_users)} users) with enough history.")

print("\n3. Building TimeSeriesDataSet...")
# Define max history (lookback) and prediction windows
max_encoder_length = 3
max_prediction_length = 1

training_dataset = TimeSeriesDataSet(
    df,
    time_idx="time_idx",
    target="monthly_expense_total",
    group_ids=["user_id"],
    min_encoder_length=1,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,

    # Static features (Things that don't change over time for the user)
    static_categoricals=["user_id", "income_type", "financial_scenario"],

    # Time-varying known features (Things we know in the future, like the time index)
    time_varying_known_reals=["time_idx"],

    # Time-varying unknown features (Things the model has to predict / learn from the past)
    time_varying_unknown_categoricals=["category", "financial_stress_level", "cash_flow_status"],
    time_varying_unknown_reals=[
        "monthly_expense_total",
        "monthly_income",
        "savings_rate",
        "credit_score",
        "debt_to_income_ratio",
        "discretionary_spending",
        "essential_spending"
    ],

    # 👇 The Normalizer that prevents the gradient from exploding! 👇
    target_normalizer=GroupNormalizer(groups=["user_id"]),

    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

train_dataloader = training_dataset.to_dataloader(train=True, batch_size=32, num_workers=0)

print("\n4. Building and Training Temporal Fusion Transformer...")
tft = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=0.005,        # Slower learning rate for precision
    hidden_size=32,             # Bigger brain
    attention_head_size=2,
    dropout=0.1,
    hidden_continuous_size=16,  # Bigger continuous brain
    loss=QuantileLoss(quantiles=[0.1, 0.5, 0.9]),
    log_interval=5,
)

trainer = pl.Trainer(
    max_epochs=50,              # Full 50 passes over the data
    accelerator="auto",
    devices=1,
    enable_model_summary=True,
)

# Start the training loop!
trainer.fit(
    tft,
    train_dataloaders=train_dataloader
)
print("✅ Done! TFT trained on Kaggle dataset.")


In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('\n── 5. Running Prediction & Evaluation ──────────────────')

# Put the model in evaluation mode so it doesn't try to train anymore
tft.eval()

# Run predictions! (We use train_dataloader since we didn't create a val_dataloader)
results = tft.predict(train_dataloader, mode="quantiles", return_y=True)

# Extract predictions (P10, P50, P90) and actual targets
preds = results.output.cpu().numpy()
y_true = results.y[0].cpu().numpy().flatten()

# We are predicting "monthly_expense_total", so we extract the quantiles
# Rounding to the nearest 10 for clean outputs
p10 = np.round(preds[:, :, 0].flatten() / 10) * 10
p50 = np.round(preds[:, :, 1].flatten() / 10) * 10
p90 = np.round(preds[:, :, 2].flatten() / 10) * 10

# Calculate Metrics
mask = y_true != 0
mape = np.mean(np.abs((y_true[mask] - p50[mask]) / y_true[mask])) * 100 if mask.any() else 0.0
rmse = np.sqrt(mean_squared_error(y_true, p50))
mae = mean_absolute_error(y_true, p50)
r2 = r2_score(y_true, p50)
coverage = np.mean((y_true >= p10) & (y_true <= p90)) * 100

print(f'\n── Kaggle Dataset Metrics ──────────────────────────────')
print(f'  MAPE         : {mape:.1f}%')
print(f'  RMSE         : {rmse:,.0f}')
print(f'  MAE          : {mae:,.0f}')
print(f'  R²           : {r2:+.3f}')
print(f'  CI Coverage  : {coverage:.1f}%')
print('────────────────────────────────────────────────────────')
